# Multi-asset portfolio valuation at scale

**Prerequisites:** selected `02_pricing/instruments/` notebooks and `05_portfolio/portfolio_construction_and_valuation.ipynb`.

This notebook builds a **representative multi-asset portfolio** spanning seventeen instrument types: **bonds** (fixed and floating), **term loans**, **revolving credit facilities**, **CDS**, **CDS indices**, **CDS tranches**, **CDS options**, **CLOs**, **ABS**, **interest rate swaps**, **swaptions**, **IR futures**, **spot equity**, **variance swaps**, **FX swaps**, and **convertible bonds**.

After valuation it extracts the **risk metrics** an investment group cares about most: **bucketed DV01**, **bucketed CS01**, **delta**, **gamma**, **vega**, **theta**, and their portfolio-level aggregations. The default smoke run uses bounded base and scale portfolios; set `FINSTACK_RUN_LARGE_NOTEBOOK_BENCH=1` for the 40 / 120-position examples. The 3,000 / 25,000-position benchmarks remain opt-in.

In [ ]:
import copy
import json
import os
import time
from datetime import date

import sys
sys.path.insert(0, "..")

from _shared.instrument_fixtures import (
    INSTRUMENT_FACTORIES,
    abs_deal,
    cds,
    cds_index,
    cds_option,
    cds_tranche,
    clo_deal,
    convertible_bond,
    fixed_bond,
    floating_bond,
    fx_swap,
    instrument_description,
    ir_future,
    irs,
    revolver,
    spot_equity,
    swaption,
    term_loan,
    variance_swap,
)
from _shared.market import DEMO_AS_OF, build_demo_market

from finstack_quant.core.currency import Currency
from finstack_quant.core.market_data import (
    BaseCorrelationCurve,
    CreditIndexData,
    DiscountCurve,
    ForwardCurve,
    FxMatrix,
    HazardCurve,
    MarketContext,
    ScalarTimeSeries,
    VolSurface,
)
from finstack_quant.portfolio import (
    Portfolio,
    PortfolioMetrics,
    PortfolioValuation,
    aggregate_metrics,
    attribute_portfolio_pnl,
    scenario_pnl,
    value_portfolio_typed,
)
from finstack_quant.scenarios import build_scenario_spec
from finstack_quant.valuations import ValuationResult
from finstack_quant.valuations.instruments import validate_instrument_json

AS_OF = DEMO_AS_OF
AS_OF_STR = AS_OF.isoformat()
RUN_LARGE_BENCHMARK = True
RUN_XL_BENCHMARK = True
RUN_ATTRIBUTION_BENCHMARK = True
BASE_POSITION_COUNT = 40 if RUN_LARGE_BENCHMARK else 3
SCALE_POSITION_COUNT = 120 if RUN_LARGE_BENCHMARK else 6

print("Imports OK.")


Imports OK.


## 1. Market data

A shared `MarketContext` provides everything the seventeen instrument types need:

| Data | ID | Used by |
|------|----|---------|
| USD OIS discount | `USD-OIS` | all USD instruments |
| EUR OIS discount | `EUR-OIS` | FX swaps (foreign leg) |
| USD SOFR 3M forward | `USD-SOFR-3M` | FRN, IRS float, revolver, swaption, IR future |
| USD IG risky discount | `USD-IG` | convertible bonds (discount) |
| USD Credit BBB discount | `USD-CREDIT-BBB` | convertible bonds (credit) |
| Corporate hazard | `CORP-HAZARD` | CDS, CDS option |
| Index hazard | `CDX-HAZ` | CDS tranche, CDS index |
| Base correlation | `CDX-BC` | CDS tranche |
| CDS spread vol surface | `CDS-SPREAD-VOL` | CDS option |
| Swaption vol surface | `USD-SWPNVOL` | swaptions |
| Equity vol surface | `TECH-VOL` | convertible bond |
| Equity spot / div prices | `TECH`, `AAPL-SPOT`, etc. | equity, convertible, variance swap |
| FX quotes | `EUR/USD` | FX swaps |

In [36]:
# Start from the canonical cross-asset demo market (USD-OIS, EUR-OIS, USD-SOFR-3M +
# fixings, CORP-HAZARD, AAPL/SPX prices + divs, EUR/USD 1.08) and add only what the
# seventeen instrument types need beyond it.
mc = build_demo_market(AS_OF)

# --- Extras: credit-risky discounting for convertible bonds ---
mc.insert(DiscountCurve(
    "USD-IG", AS_OF,
    [(0.0, 1.0), (1.0, 0.945), (3.0, 0.84), (5.0, 0.75), (10.0, 0.58)],
    day_count="act_365f",
))

mc.insert(DiscountCurve(
    "USD-CREDIT-BBB", AS_OF,
    [(0.0, 1.0), (1.0, 0.935), (3.0, 0.82), (5.0, 0.72), (10.0, 0.52)],
    day_count="act_365f",
))

# --- Extras: index credit for CDS tranches and CDS indices ---
mc.insert(HazardCurve(
    "CDX-HAZ", AS_OF,
    [(1.0, 0.01), (3.0, 0.015), (5.0, 0.02), (10.0, 0.025)],
    recovery_rate=0.40,
    par_spreads=[(1.0, 60.347022), (3.0, 80.603057),
                 (5.0, 95.744830), (10.0, 118.825427)],
))

base_correlation = BaseCorrelationCurve(
    "CDX-BC",
    [(3.0, 0.30), (7.0, 0.30), (10.0, 0.30), (15.0, 0.30), (30.0, 0.30), (100.0, 0.30)],
)
mc.insert(base_correlation)
mc.insert_credit_index(
    "CDX.NA.IG.HAZARD",
    CreditIndexData(125, 0.40, mc.get_hazard("CDX-HAZ"), base_correlation),
)

# --- Extras: volatility surfaces ---
mc.insert(VolSurface(
    "CDS-SPREAD-VOL", [0.25, 0.5, 1.0, 2.0],
    [50.0, 100.0, 150.0, 200.0, 300.0], [0.30] * 20,
))
mc.insert(VolSurface(
    "TECH-VOL", [0.25, 0.5, 1.0, 2.0, 5.0],
    [20.0, 30.0, 40.0, 50.0, 60.0], [0.35] * 25,
))
mc.insert(VolSurface(
    "USD-SWPNVOL", [0.25, 0.5, 1.0, 2.0, 5.0],
    [0.02, 0.03, 0.04, 0.05, 0.06], [0.20] * 25,
))

# --- Extras: convertible underlier and variance-swap inputs ---
mc.insert_price("TECH", 42.0, "USD")
mc.insert_price("SPX_IMPL_VOL", 0.20)

spx_history = [
    (AS_OF, 5200.0),
    (date(2025, 1, 16), 5096.0),
    (date(2025, 1, 17), 5096.0),
]
mc.insert_series(ScalarTimeSeries("SPX", spx_history, currency="USD"))

MARKET_JSON = mc.to_json()
curve_ids = [
    "USD-OIS", "EUR-OIS", "USD-SOFR-3M", "USD-IG", "USD-CREDIT-BBB",
    "CORP-HAZARD", "CDX-HAZ", "CDX-BC",
]
print(f"Market snapshot ready — {len(curve_ids)} curves, 3 surfaces, 6 prices")
print("Curve IDs:", curve_ids)

Market snapshot ready — 8 curves, 3 surfaces, 6 prices
Curve IDs: ['USD-OIS', 'EUR-OIS', 'USD-SOFR-3M', 'USD-IG', 'USD-CREDIT-BBB', 'CORP-HAZARD', 'CDX-HAZ', 'CDX-BC']


## 2. Base instrument templates

Each helper returns a `(instrument_id, instrument_spec)` tuple. The `instrument_spec` is the tagged `{"type": ..., "spec": ...}` dict ready to embed in a portfolio position.

In [37]:
# One representative payload per asset class stays visible at the lesson boundary.
WIRE_CONTRACT_EXAMPLES = {
    "fixed_income": fixed_bond(0)[1],
    "credit": cds(0)[1],
    "structured_credit": clo_deal(0)[1],
    "rates": irs(0)[1],
    "equity": spot_equity(0)[1],
    "fx": fx_swap(0)[1],
}
for asset_class, payload in WIRE_CONTRACT_EXAMPLES.items():
    print(f"\n{asset_class} wire payload:")
    print(json.dumps(payload, indent=2)[:800], "…")



fixed_income wire payload:
{
  "type": "bond",
  "spec": {
    "id": "BOND-FIXED-0",
    "notional": {
      "amount": "1000000",
      "currency": "USD"
    },
    "issue_date": "2024-01-15",
    "maturity": "2028-01-15",
    "discount_curve_id": "USD-OIS",
    "accrual_method": "Linear",
    "settlement_days": 1,
    "ex_coupon_days": 0,
    "cashflow_spec": {
      "Fixed": {
        "coupon_type": "Cash",
        "freq": {
          "count": 6,
          "unit": "months"
        },
        "dc": "Thirty360",
        "bdc": "following",
        "calendar_id": "weekends_only",
        "end_of_month": false,
        "payment_lag_days": 0,
        "rate": "0.04",
        "stub": "None"
      }
    },
    "call_put": null,
    "credit_curve_id": null,
    "attributes": {
      "tags": [
        "fixed-income"
      ] …

credit wire payload:
{
  "type": "credit_default_swap",
  "spec": {
    "id": "CDS-0",
    "notional": {
      "amount": 10000000.0,
      "currency": "USD"
    },
    

## 3. Validate base instruments

Confirm each factory produces valid JSON before embedding in a portfolio.

In [38]:
factories = INSTRUMENT_FACTORIES

for name, factory in factories:
    iid, spec = factory(0)
    validate_instrument_json(json.dumps(spec))
    print(f"  {name:20s} -> {iid:20s}  OK")


  fixed_bond           -> BOND-FIXED-0          OK
  floating_bond        -> BOND-FRN-0            OK
  term_loan            -> TL-0                  OK
  revolver             -> RCF-0                 OK
  cds                  -> CDS-0                 OK
  cds_index            -> CDX-IDX-0             OK
  cds_tranche          -> CDX-TR-0              OK
  cds_option           -> CDSOPT-0              OK
  clo_deal             -> CLO-0                 OK
  abs_deal             -> ABS-0                 OK
  irs                  -> IRS-0                 OK
  swaption             -> SWPN-0                OK
  ir_future            -> IRF-0                 OK
  spot_equity          -> EQ-0                  OK
  variance_swap        -> VARSWAP-0             OK
  fx_swap              -> FXSWAP-0              OK
  convertible_bond     -> CB-0                  OK


## 4. Build the configured base portfolio

The configured portfolio uses a representative instrument mix across fund entities. Its size is selected in the import cell so smoke and opt-in benchmark runs report the same source of truth.

In [39]:
ALLOCATION = [
    (fixed_bond,       0.10),
    (floating_bond,    0.08),
    (term_loan,        0.06),
    (revolver,         0.04),
    (cds,              0.10),
    (cds_index,        0.04),
    (cds_tranche,      0.04),
    (cds_option,       0.04),
    (clo_deal,         0.04),
    (abs_deal,         0.04),
    (irs,              0.10),
    (swaption,         0.05),
    (ir_future,        0.05),
    (spot_equity,      0.05),
    (variance_swap,    0.04),
    (fx_swap,          0.04),
    (convertible_bond, 0.09),
]

if not RUN_LARGE_BENCHMARK:
    # Keep the default run representative and bounded with liquid, lightweight
    # fixed-income, rates, equity, FX, and volatility instruments. Credit and
    # structured-credit coverage remains visible above and in opt-in scale runs.
    ALLOCATION = [
        (fixed_bond, 1 / 6),
        (irs, 1 / 6),
        (spot_equity, 1 / 6),
        (fx_swap, 1 / 6),
        (variance_swap, 1 / 6),
        (floating_bond, 1 / 6),
    ]



def build_positions(target_size: int) -> list[dict]:
    positions = []
    pos_id = 0
    for factory, weight in ALLOCATION:
        count = max(1, int(target_size * weight))
        for i in range(count):
            if pos_id >= target_size:
                break
            iid, spec = factory(i)
            iid_unique = f"{iid}-P{pos_id}"
            spec_copy = copy.deepcopy(spec)
            spec_copy["spec"]["id"] = iid_unique
            _fixup_structured_ids(spec_copy, iid, iid_unique)
            desc = instrument_description(spec_copy)
            positions.append({
                "position_id": f"POS-{pos_id} ({desc})",
                "entity_id": f"FUND-{(pos_id % 3) + 1}",
                "instrument_id": iid_unique,
                "instrument_spec": spec_copy,
                "quantity": 1.0,
                "unit": "units",
            })
            pos_id += 1
        if pos_id >= target_size:
            break

    while pos_id < target_size:
        iid, spec = fixed_bond(pos_id)
        iid_unique = f"{iid}-P{pos_id}"
        spec_copy = copy.deepcopy(spec)
        spec_copy["spec"]["id"] = iid_unique
        desc = instrument_description(spec_copy)
        positions.append({
            "position_id": f"POS-{pos_id} ({desc})",
            "entity_id": f"FUND-{(pos_id % 3) + 1}",
            "instrument_id": iid_unique,
            "instrument_spec": spec_copy,
            "quantity": 1.0,
            "unit": "units",
        })
        pos_id += 1

    return positions


def _fixup_structured_ids(spec: dict, old_prefix: str, new_id: str):
    if spec.get("type") != "structured_credit":
        return
    inner = spec["spec"]
    inner["pool"]["id"] = f"{new_id}-POOL"
    for j, tr in enumerate(inner.get("tranches", {}).get("tranches", [])):
        tr["id"] = f"{new_id}-T{j}"
    for j, asset in enumerate(inner.get("pool", {}).get("assets", [])):
        asset["id"] = f"{new_id}-A{j}"


print("Position builder ready.")

Position builder ready.


In [40]:
def make_portfolio_json(portfolio_id: str, num_positions: int) -> str:
    positions = build_positions(num_positions)
    entity_ids = sorted({p["entity_id"] for p in positions})
    return json.dumps({
        "id": portfolio_id,
        "as_of": AS_OF_STR,
        "base_ccy": "USD",
        "entities": {eid: {"id": eid} for eid in entity_ids},
        "positions": positions,
    })


base_portfolio_json = make_portfolio_json("MULTI-ASSET-BASE", BASE_POSITION_COUNT)
base_obj = json.loads(base_portfolio_json)
print(f"Base portfolio: {len(base_obj['positions'])} positions, {len(base_obj['entities'])} entities")

type_counts: dict[str, int] = {}
for p in base_obj["positions"]:
    t = p["instrument_spec"]["type"]
    type_counts[t] = type_counts.get(t, 0) + 1
for t, c in sorted(type_counts.items()):
    print(f"  {t:25s}: {c}")

Base portfolio: 40 positions, 3 entities
  bond                     : 13
  cds_index                : 1
  cds_option               : 1
  cds_tranche              : 1
  convertible_bond         : 3
  credit_default_swap      : 4
  equity                   : 2
  fx_swap                  : 1
  interest_rate_future     : 2
  interest_rate_swap       : 4
  revolving_credit         : 1
  structured_credit        : 2
  swaption                 : 2
  term_loan                : 2
  variance_swap            : 1


## 5. Value the base portfolio

In [ ]:
base_portfolio = Portfolio.from_spec(base_portfolio_json)
t0 = time.perf_counter()
pv_base = value_portfolio_typed(base_portfolio, mc)
elapsed_base = time.perf_counter() - t0

val_result = pv_base.to_json()
# Raw parse retained only for per-position detail used by later cells:
# PortfolioValuation exposes the base-currency total and position count,
# but has no typed per-position accessor.
val_obj = json.loads(val_result)
total_pv = pv_base.total_value
n_priced = len(pv_base)
print(f"Base portfolio PV: {total_pv:,.2f} USD  ({n_priced} positions valued in {elapsed_base:.3f}s)")

### Per-position breakdown

In [ ]:
for pos_id, pv_info in list(val_obj["position_values"].items())[:12]:
    pv = float(pv_info["value_base"]["amount"])
    print(f"  {pos_id:<36s} PV = {pv:>18,.2f} USD")
if n_priced > 12:
    print(f"  ... ({n_priced - 12} more positions)")

  POS-0 (Fixed 3Y 4.0%)                PV =         980,480.04 USD
  POS-1 (IRS Pay 3Y 4.0%)              PV =         231,157.51 USD
  POS-2 (AAPL 100sh)                   PV =          18,500.00 USD


### Per-position risk sensitivities

`value_portfolio` automatically computes the standard portfolio metric set for every position:
**DV01**, **bucketed DV01** (key-rate), **CS01**, **bucketed CS01**, **delta**, **gamma**, **vega**, **rho**, **pv01**, and **theta**. Not every instrument supports every metric -- the engine silently omits inapplicable ones.

In [ ]:
RISK_KEYS = ["dv01", "cs01", "theta", "delta", "gamma", "vega", "rho", "pv01"]

header = f"{'Position':<36s} {'Type':<24s}" + "".join(f"{k:>12s}" for k in RISK_KEYS)
print(header)
print("-" * len(header))

for pid, pval in val_obj["position_values"].items():
    vr = pval.get("valuation_result")
    if not vr:
        continue
    measures = vr.get("measures", {})
    itype = pval.get("instrument_type", "")
    if not itype:
        for p in base_obj["positions"]:
            if p["position_id"] == pid:
                itype = p["instrument_spec"]["type"]
                break
    row = f"{pid:<36s} {itype:<24s}"
    for k in RISK_KEYS:
        v = measures.get(k)
        row += f"{v:>12.2f}" if v is not None else f"{'--':>12s}"
    print(row)

Position                             Type                            dv01        cs01       theta       delta       gamma        vega         rho        pv01
-------------------------------------------------------------------------------------------------------------------------------------------------------------
POS-0 (Fixed 3Y 4.0%)                bond                         -280.39     -273.64    20119.80          --          --        0.00          --          --
POS-1 (IRS Pay 3Y 4.0%)              interest_rate_swap           2690.06          --       28.25          --          --          --          --     2760.76
POS-2 (AAPL 100sh)                   equity                          0.00          --        0.00      100.00          --          --          --          --


### Bucketed DV01 -- rate risk by tenor

Key-rate DV01 decomposes the parallel DV01 across standard tenor buckets for every curve the position depends on. The keys follow the pattern `bucketed_dv01::<curve_id>::<tenor>`.

In [ ]:
def pivot_buckets(
    val_obj: dict,
    base_obj: dict,
    base_metric: str,
) -> tuple[list[tuple[str, ...]], dict]:
    """Collect decoded bucket components through ValuationResult.metric_series."""
    all_keys: set[tuple[str, ...]] = set()
    rows = {}
    position_types = {
        position["position_id"]: position["instrument_spec"]["type"]
        for position in base_obj["positions"]
    }
    for pid, pval in val_obj["position_values"].items():
        payload = pval.get("valuation_result")
        if not payload:
            continue
        result = ValuationResult.from_json(json.dumps(payload))
        buckets = {tuple(components): value for components, value in result.metric_series(base_metric)}
        if not buckets:
            continue
        rows[(pid, position_types.get(pid, ""))] = buckets
        all_keys.update(buckets)
    return list(all_keys), rows


def tenor_sort_key(components: tuple[str, ...]) -> tuple[str, float]:
    """Sort decoded metric components by curve then numeric tenor."""
    curve = components[0] if len(components) > 1 else ""
    tenor = components[-1] if components else ""
    multiplier = {"m": 1 / 12, "y": 1}.get(tenor[-1], 1) if tenor else 0
    try:
        number = float(tenor[:-1]) * multiplier
    except (ValueError, IndexError):
        number = 999.0
    return (curve, number)


sorted_keys, dv01_rows = pivot_buckets(val_obj, base_obj, "bucketed_dv01")
sorted_keys.sort(key=tenor_sort_key)

short_labels = ["::".join(components) for components in sorted_keys]
header = f"{'Position':<36s} {'Type':<20s}" + "".join(f"{label:>18s}" for label in short_labels)
print("Bucketed DV01 (USD/bp)")
print(header)
print("-" * len(header))
for (pid, itype), buckets in dv01_rows.items():
    row = f"{pid:<36s} {itype:<20s}"
    for components in sorted_keys:
        value = buckets.get(components)
        row += f"{value:>18.4f}" if value is not None else f"{'':>18s}"
    print(row)

Bucketed DV01 (USD/bp)
Position                             Type                       USD-OIS::3m       USD-OIS::6m       USD-OIS::1y       USD-OIS::2y       USD-OIS::3y       USD-OIS::5y       USD-OIS::7y      USD-OIS::10y      USD-OIS::15y      USD-OIS::20y      USD-OIS::30y   USD-SOFR-3M::3m   USD-SOFR-3M::6m   USD-SOFR-3M::1y   USD-SOFR-3M::2y   USD-SOFR-3M::3y   USD-SOFR-3M::5y   USD-SOFR-3M::7y  USD-SOFR-3M::10y  USD-SOFR-3M::15y  USD-SOFR-3M::20y  USD-SOFR-3M::30y
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
POS-0 (Fixed 3Y 4.0%)                bond                           -

### Bucketed CS01 -- credit spread risk by tenor

Analogous to DV01 but bumping hazard/credit curves. Keys follow `bucketed_cs01::<curve_id>::<tenor>`. Bond CS01 uses z-spread bump and keys off the instrument ID.

In [ ]:
sorted_cs_keys, cs01_rows = pivot_buckets(val_obj, base_obj, "bucketed_cs01")
sorted_cs_keys.sort(key=tenor_sort_key)

if cs01_rows:
    short_cs = ["::".join(components) for components in sorted_cs_keys]
    header = f"{'Position':<36s} {'Type':<20s}" + "".join(f"{label:>18s}" for label in short_cs)
    print("Bucketed CS01 (USD/bp)")
    print(header)
    print("-" * len(header))
    for (pid, itype), buckets in cs01_rows.items():
        row = f"{pid:<36s} {itype:<20s}"
        for components in sorted_cs_keys:
            value = buckets.get(components)
            row += f"{value:>18.4f}" if value is not None else f"{'':>18s}"
        print(row)
else:
    print("No bucketed CS01 data (no credit instruments in base portfolio).")

No bucketed CS01 data (no credit instruments in base portfolio).


## 6. Portfolio-level risk report

`aggregate_metrics` sums all summable sensitivities across positions (with FX conversion when needed) and provides breakdowns by entity.

In [ ]:
agg_json = aggregate_metrics(pv_base, "USD", mc, AS_OF_STR)
agg = json.loads(agg_json)
agg_metrics = PortfolioMetrics.from_json(agg_json)

print(f"Portfolio Risk Summary (base portfolio, {n_priced} positions)")
print("=" * 60)

headline_metrics = ["dv01", "cs01", "theta", "delta", "gamma", "vega", "rho", "pv01"]
for metric in headline_metrics:
    info = agg["aggregated"].get(metric)
    if info:
        print(f"  {metric:<12s}  {info['total']:>18,.2f} USD")
    else:
        print(f"  {metric:<12s}  {'n/a':>18s}")

Portfolio Risk Summary (base portfolio, 3 positions)
  dv01                    2,409.66 USD
  cs01                     -273.64 USD
  theta                  20,148.05 USD
  delta                     100.00 USD
  gamma                        n/a
  vega                        0.00 USD
  rho                          n/a
  pv01                    2,760.76 USD


### Aggregated bucketed DV01 by tenor

Portfolio-wide rate exposure summed across all positions.

In [ ]:
def print_agg_buckets(metrics: PortfolioMetrics, base_metric: str, title: str) -> None:
    """Print Rust-decoded aggregate bucket series with entity breakdown."""
    series = sorted(metrics.metric_series(base_metric), key=lambda item: tenor_sort_key(tuple(item[0])))
    if not series:
        print(f"No {title} data in aggregation.")
        return

    entities = sorted({entity for _, _, by_entity in series for entity in by_entity})
    header = f"{'Bucket':<30s} {'Total':>14s}" + "".join(f"{entity:>14s}" for entity in entities)
    print(title)
    print(header)
    print("-" * len(header))
    for components, total, by_entity in series:
        label = "::".join(components)
        row = f"{label:<30s} {total:>14,.2f}"
        for entity in entities:
            row += f"{by_entity.get(entity, 0):>14,.2f}"
        print(row)
    print()


print_agg_buckets(agg_metrics, "bucketed_dv01", "Aggregated Bucketed DV01 (USD/bp)")

Aggregated Bucketed DV01 (USD/bp)
Bucket                                  Total        FUND-1        FUND-2        FUND-3
---------------------------------------------------------------------------------------
USD-OIS::3m                             -0.37         -0.01         -0.37          0.00
USD-OIS::6m                             -3.55         -0.81         -2.74          0.00
USD-OIS::1y                             -8.47         -3.09         -5.39          0.00
USD-OIS::2y                            -24.45         -7.09        -17.36          0.00
USD-OIS::3y                           -276.78       -269.17         -7.61          0.00
USD-OIS::5y                              2.02         -0.23          2.25          0.00
USD-OIS::7y                              0.00          0.00          0.00          0.00
USD-OIS::10y                            -0.11          0.00         -0.11          0.00
USD-OIS::15y                             0.00          0.00          0.00          0.0

### Aggregated bucketed CS01 by tenor

Portfolio-wide credit spread exposure across hazard curves.

In [ ]:
print_agg_buckets(agg_metrics, "bucketed_cs01", "Aggregated Bucketed CS01 (USD/bp)")

No Aggregated Bucketed CS01 (USD/bp) data in aggregation.


### Risk by entity

Break down headline sensitivities by fund entity to see risk concentration.

In [ ]:
entities = sorted({e for info in agg["aggregated"].values() for e in info.get("by_entity", {})})
header = f"{'Metric':<12s} {'Total':>16s}" + "".join(f"{e:>16s}" for e in entities)
print("Risk by Entity (USD)")
print(header)
print("-" * len(header))
for m in headline_metrics:
    info = agg["aggregated"].get(m)
    if not info:
        continue
    row = f"{m:<12s} {info['total']:>16,.2f}"
    for e in entities:
        row += f"{info['by_entity'].get(e, 0):>16,.2f}"
    print(row)

Risk by Entity (USD)
Metric                  Total          FUND-1          FUND-2          FUND-3
-----------------------------------------------------------------------------
dv01                 2,409.66         -280.39        2,690.06            0.00
cs01                  -273.64         -273.64            0.00            0.00
theta               20,148.05       20,119.80           28.25            0.00
delta                  100.00            0.00            0.00          100.00
vega                     0.00            0.00            0.00            0.00
pv01                 2,760.76            0.00        2,760.76            0.00


## 7. Value the configured scale portfolio

In [ ]:
portfolio_500_json = make_portfolio_json("MULTI-ASSET-SCALE", SCALE_POSITION_COUNT)
obj_500 = json.loads(portfolio_500_json)

type_counts_500: dict[str, int] = {}
for p in obj_500["positions"]:
    t = p["instrument_spec"]["type"]
    type_counts_500[t] = type_counts_500.get(t, 0) + 1
print(f"{SCALE_POSITION_COUNT}-position portfolio instrument mix:")
for t, c in sorted(type_counts_500.items()):
    print(f"  {t:25s}: {c:4d}  ({100*c/SCALE_POSITION_COUNT:.0f}%)")

portfolio_500 = Portfolio.from_spec(portfolio_500_json)
t0 = time.perf_counter()
pv_500 = value_portfolio_typed(portfolio_500, mc)
elapsed_500 = time.perf_counter() - t0

val_500 = pv_500.to_json()
total_500 = pv_500.total_value
n_500 = len(pv_500)
print(f"\nTotal PV: {total_500:,.2f} USD")
print(f"Valued {n_500} positions in {elapsed_500:.3f}s  ({n_500/elapsed_500:,.0f} positions/sec)")

6-position portfolio instrument mix:
  bond                     :    2  (33%)
  equity                   :    1  (17%)
  fx_swap                  :    1  (17%)
  interest_rate_swap       :    1  (17%)
  variance_swap            :    1  (17%)

Total PV: 2,246,532.58 USD
Valued 6 positions in 0.004s  (1,431 positions/sec)


## 8. Optional PV-only scale to 3,000 positions

This throughput run requests `metrics=[]`, so it measures portfolio PV and FX aggregation rather than multiplying the workload by the full finite-difference risk set. The configured-scale run above remains the full-risk benchmark.

In [ ]:
if RUN_LARGE_BENCHMARK:
    portfolio_3000_json = make_portfolio_json("MULTI-ASSET-3000", 3000)
    obj_3000 = json.loads(portfolio_3000_json)

    type_counts_3k: dict[str, int] = {}
    for p in obj_3000["positions"]:
        t = p["instrument_spec"]["type"]
        type_counts_3k[t] = type_counts_3k.get(t, 0) + 1
    print("3,000-position portfolio instrument mix:")
    for t, c in sorted(type_counts_3k.items()):
        print(f"  {t:25s}: {c:4d}  ({100*c/3000:.0f}%)")

    portfolio_3000 = Portfolio.from_spec(portfolio_3000_json)
    t0 = time.perf_counter()
    pv_3000 = value_portfolio_typed(portfolio_3000, mc, metrics=[])
    elapsed_3000 = time.perf_counter() - t0

    val_3000 = pv_3000.to_json()
    total_3000 = pv_3000.total_value
    n_3000 = len(pv_3000)
    print(f"\nTotal PV: {total_3000:,.2f} USD")
    print(f"Valued {n_3000} positions in {elapsed_3000:.3f}s  ({n_3000/elapsed_3000:,.0f} positions/sec)")
else:
    portfolio_3000_json = portfolio_500_json
    obj_3000 = obj_500
    val_3000 = val_500
    pv_3000 = pv_500
    total_3000 = total_500
    n_3000 = n_500
    elapsed_3000 = 0.0
    print("3,000-position benchmark skipped; set FINSTACK_RUN_LARGE_NOTEBOOK_BENCH=1 to run it.")

3,000-position benchmark skipped; set FINSTACK_RUN_LARGE_NOTEBOOK_BENCH=1 to run it.


## 9. Optional PV-only scale to 25,000 positions

As above, this is a PV-throughput run (`metrics=[]`). Full bucketed credit risk is intentionally demonstrated on the smaller configured-scale portfolio.

In [ ]:
if RUN_XL_BENCHMARK:
    portfolio_25k_json = make_portfolio_json("MULTI-ASSET-25K", 25_000)
    obj_25k = json.loads(portfolio_25k_json)

    type_counts_25k: dict[str, int] = {}
    for p in obj_25k["positions"]:
        t = p["instrument_spec"]["type"]
        type_counts_25k[t] = type_counts_25k.get(t, 0) + 1
    print("25,000-position portfolio instrument mix:")
    for t, c in sorted(type_counts_25k.items()):
        print(f"  {t:25s}: {c:5d}  ({100 * c / 25_000:.0f}%)")

    portfolio_25k = Portfolio.from_spec(portfolio_25k_json)
    t0 = time.perf_counter()
    pv_25k = value_portfolio_typed(portfolio_25k, mc, metrics=[])
    elapsed_25k = time.perf_counter() - t0

    val_25k = pv_25k.to_json()
    total_25k = pv_25k.total_value
    n_25k = len(pv_25k)
    print(f"\nTotal PV: {total_25k:,.2f} USD")
    print(f"Valued {n_25k} positions in {elapsed_25k:.3f}s  ({n_25k / elapsed_25k:,.0f} positions/sec)")
else:
    portfolio_25k_json = portfolio_3000_json
    obj_25k = obj_3000
    val_25k = val_3000
    pv_25k = pv_3000
    total_25k = total_3000
    n_25k = n_3000
    elapsed_25k = 0.0
    print("25,000-position benchmark skipped; set FINSTACK_RUN_XL_NOTEBOOK_BENCH=1 to run it.")

25,000-position benchmark skipped; set FINSTACK_RUN_XL_NOTEBOOK_BENCH=1 to run it.


## 10. Performance summary

In [ ]:
print("Portfolio valuation throughput (full risk at configured scale; PV-only at large scale)")
print("=" * 75)
rows = [
    (f"Base ({n_priced})", n_priced, elapsed_base, total_pv),
    (f"Configured scale ({n_500})", n_500, elapsed_500, total_500),
]
if RUN_LARGE_BENCHMARK:
    rows.append(("PV-only (3,000)", n_3000, elapsed_3000, total_3000))
else:
    print("  Large (3,000) benchmark skipped by default")
if RUN_XL_BENCHMARK:
    rows.append(("PV-only (25,000)", n_25k, elapsed_25k, total_25k))
else:
    print("  XL (25,000) benchmark skipped by default")

for label, n, elapsed, pv in rows:
    rate = n / elapsed if elapsed > 0 else float("inf")
    print(f"  {label:20s}  {n:6d} positions  {elapsed:7.3f}s  {rate:>8,.0f} pos/s  PV={pv:>18,.2f} USD")

Portfolio valuation throughput
  Large (3,000) benchmark skipped by default
  XL (25,000) benchmark skipped by default
  Base (3)                   3 positions    0.002s     1,874 pos/s  PV=      1,230,137.56 USD
  Configured scale (6)       6 positions    0.004s     1,431 pos/s  PV=      2,246,532.58 USD


## 11. 1-day P&L attribution

Simulate an overnight market move (rates +5 bp, credit spreads +10 bp, equity -2%, vol +1 pt, FX EUR/USD -0.5%) and attribute the full book with Rust's `attribute_portfolio_pnl` **MetricsBased** methodology. This method supports the complete instrument mix while keeping valuation, factor mapping, cross-currency translation, aggregation, and reconciliation in Rust.

Python reads typed aggregate `Money` fields and uses the nested position JSON only for drill-down display.

In [ ]:
AS_OF_T1 = "2025-01-16"
T1 = date(2025, 1, 16)

# The T1 snapshot mirrors the T0 knot structure exactly, so the attribution sees a
# pure market move rather than a change in curve resolution. Fixings and the SPX
# history are read back off the T0 context instead of being rebuilt.
mc_t1 = MarketContext()

mc_t1.insert(DiscountCurve(
    "USD-OIS", T1,
    [(0.0, 1.0), (0.25, 0.9885), (0.5, 0.9770), (1.0, 0.9540), (2.0, 0.9085),
     (3.0, 0.8680), (5.0, 0.7975), (10.0, 0.6465)],
    day_count="act_365f",
))

mc_t1.insert(DiscountCurve(
    "EUR-OIS", T1,
    [(0.0, 1.0), (1.0, 0.9690), (3.0, 0.9080), (5.0, 0.8475)],
    day_count="act_365f",
))

mc_t1.insert(ForwardCurve(
    "USD-SOFR-3M", 0.25,
    knots=[(0.0, 0.0455), (1.0, 0.0475), (3.0, 0.0495), (10.0, 0.0525)],
    base_date=T1,
    day_count="act_360",
))

mc_t1.insert(DiscountCurve(
    "USD-IG", T1,
    [(0.0, 1.0), (1.0, 0.9435), (3.0, 0.8375), (5.0, 0.7465), (10.0, 0.5755)],
    day_count="act_365f",
))

mc_t1.insert(DiscountCurve(
    "USD-CREDIT-BBB", T1,
    [(0.0, 1.0), (1.0, 0.9330), (3.0, 0.8165), (5.0, 0.7155), (10.0, 0.5145)],
    day_count="act_365f",
))

mc_t1.insert(HazardCurve(
    "CORP-HAZARD", T1,
    [(1.0, 0.021), (3.0, 0.025), (5.0, 0.029), (10.0, 0.033)],
    recovery_rate=0.40,
    par_spreads=[(1.0, 126.639125), (3.0, 144.717567),
                 (5.0, 157.417265), (10.0, 176.063391)],
))

mc_t1.insert(HazardCurve(
    "CDX-HAZ", T1,
    [(1.0, 0.011), (3.0, 0.016), (5.0, 0.021), (10.0, 0.026)],
    recovery_rate=0.40,
    par_spreads=[(1.0, 70.347022), (3.0, 90.603057),
                 (5.0, 105.744830), (10.0, 128.825427)],
))

base_correlation_t1 = BaseCorrelationCurve(
    "CDX-BC",
    [(3.0, 0.30), (7.0, 0.30), (10.0, 0.30), (15.0, 0.30), (30.0, 0.30), (100.0, 0.30)],
)
mc_t1.insert(base_correlation_t1)
mc_t1.insert_credit_index(
    "CDX.NA.IG.HAZARD",
    CreditIndexData(125, 0.40, mc_t1.get_hazard("CDX-HAZ"), base_correlation_t1),
)

mc_t1.insert(VolSurface(
    "CDS-SPREAD-VOL", [0.25, 0.5, 1.0, 2.0],
    [50.0, 100.0, 150.0, 200.0, 300.0], [0.31] * 20,
))
mc_t1.insert(VolSurface(
    "TECH-VOL", [0.25, 0.5, 1.0, 2.0, 5.0],
    [20.0, 30.0, 40.0, 50.0, 60.0], [0.36] * 25,
))
mc_t1.insert(VolSurface(
    "USD-SWPNVOL", [0.25, 0.5, 1.0, 2.0, 5.0],
    [0.02, 0.03, 0.04, 0.05, 0.06], [0.21] * 25,
))

mc_t1.insert_price("TECH", 41.16, "USD")
mc_t1.insert_price("AAPL-SPOT", 181.30, "USD")
mc_t1.insert_price("AAPL-DIV", 0.005)
mc_t1.insert_price("SPX-SPOT", 5096.0, "USD")
mc_t1.insert_price("SPX-DIV", 0.015)
mc_t1.insert_price("SPX_IMPL_VOL", 0.21)

# Reuse the T0 series rather than rebuilding them.
mc_t1.insert_series(mc.get_series("FIXING:USD-SOFR-3M"))
mc_t1.insert_series(mc.get_series("SPX"))

fx_matrix_t1 = FxMatrix()
fx_matrix_t1.set_quote(Currency("EUR"), Currency("USD"), 1.0746)
mc_t1.insert_fx(fx_matrix_t1)

MARKET_T1_JSON = mc_t1.to_json()
print("T1 market snapshot ready (rates +5bp, credit +10bp, equity -2%, vol +1pt, EUR/USD -0.5%)")

T1 market snapshot ready (rates +5bp, credit +10bp, equity -2%, vol +1pt, EUR/USD -0.5%)


In [ ]:
FACTOR_COLS = [
    ("carry", "Carry"),
    ("rates_curves_pnl", "Rates"),
    ("credit_curves_pnl", "Credit"),
    ("inflation_curves_pnl", "Inflation"),
    ("correlations_pnl", "Correlations"),
    ("fx_pnl", "FX"),
    ("fx_translation_pnl", "FX Translation"),
    ("cross_factor_pnl", "Cross"),
    ("vol_pnl", "Vol"),
    ("model_params_pnl", "Model Params"),
    ("market_scalars_pnl", "Scalars"),
    ("residual", "Residual"),
]
POSITION_FACTOR_COLS = [
    (field, label_name)
    for field, label_name in FACTOR_COLS
    if field != "fx_translation_pnl"
]


def _money_amount(payload: dict) -> float:
    return float(payload["amount"])


def run_portfolio_attribution(portfolio_obj: dict, label: str) -> dict:
    """Run one Rust portfolio attribution and prepare nested detail for display."""
    portfolio = Portfolio.from_spec(json.dumps(portfolio_obj))
    started = time.perf_counter()
    attribution = attribute_portfolio_pnl(
        portfolio,
        mc,
        mc_t1,
        AS_OF_STR,
        AS_OF_T1,
        "MetricsBased",
    )
    elapsed = time.perf_counter() - started

    if attribution.result_invalid:
        raise RuntimeError(f"{label}: Rust flagged the portfolio attribution invalid")
    reconciliation = attribution.reconciliation_check(1.0e-6)
    if not reconciliation["is_reconciled"]:
        raise RuntimeError(f"{label}: attribution failed reconciliation: {reconciliation}")

    totals = {
        "total_pnl": attribution.total_pnl.amount,
        "Carry": attribution.carry.amount,
        "Rates": attribution.rates_curves_pnl.amount,
        "Credit": attribution.credit_curves_pnl.amount,
        "Inflation": attribution.inflation_curves_pnl.amount,
        "Correlations": attribution.correlations_pnl.amount,
        "FX": attribution.fx_pnl.amount,
        "FX Translation": attribution.fx_translation_pnl.amount,
        "Cross": attribution.cross_factor_pnl.amount,
        "Vol": attribution.vol_pnl.amount,
        "Model Params": attribution.model_params_pnl.amount,
        "Scalars": attribution.market_scalars_pnl.amount,
        "Residual": attribution.residual.amount,
    }
    position_types = {
        position["position_id"]: position["instrument_spec"]["type"]
        for position in portfolio_obj["positions"]
    }
    position_attrs = []
    for position_id, payload in json.loads(attribution.by_position_json()).items():
        row = {
            "position_id": position_id,
            "type": position_types[position_id],
            "total_pnl": _money_amount(payload["total_pnl"]),
        }
        for field, label_name in POSITION_FACTOR_COLS:
            row[label_name] = _money_amount(payload[field])
        position_attrs.append(row)

    count = len(portfolio_obj["positions"])
    print(f"{label}: {count} positions attributed in {elapsed:.3f}s ({count / elapsed:,.0f} pos/s)")
    return {
        "attribution": attribution,
        "totals": totals,
        "positions": position_attrs,
        "elapsed": elapsed,
        "reconciliation": reconciliation,
    }

base_attr_count = len(base_obj["positions"])
if RUN_ATTRIBUTION_BENCHMARK:
    base_attr = run_portfolio_attribution(base_obj, f"Base ({n_priced})")
else:
    base_attr = None
    print(
        "Portfolio attribution benchmark skipped; set "
        "FINSTACK_RUN_ATTRIBUTION_NOTEBOOK_BENCH=1 to run it."
    )


Portfolio attribution benchmark skipped; set FINSTACK_RUN_ATTRIBUTION_NOTEBOOK_BENCH=1 to run it.


### Base portfolio P&L decomposition

In [ ]:
def print_attribution_summary(result: dict, title: str) -> None:
    totals = result["totals"]
    reconciliation = result["reconciliation"]
    print(f"\n{title}")
    print("=" * 70)
    print(f"  {'Total P&L':<20s}  {totals['total_pnl']:>18,.2f} USD")
    print(f"  {'-' * 56}")
    for _, label_name in FACTOR_COLS:
        print(f"  {label_name:<20s}  {totals[label_name]:>18,.2f} USD")
    print(f"  {'-' * 56}")
    print(f"  {'Reconciliation':<20s}  {reconciliation['total_residual']:>18.8f} USD")
    print(f"  {'Reconciled':<20s}  {str(reconciliation['is_reconciled']):>18s}")

if base_attr is not None:
    print_attribution_summary(
        base_attr, f"Portfolio Attribution ({base_attr_count} positions)"
    )
else:
    print("Base portfolio attribution display skipped.")


Base portfolio attribution display skipped.


### Per-position attribution detail (base portfolio)

In [ ]:
if base_attr is None:
    print("Per-position attribution detail skipped.")
else:
    position_cols = ["Total"] + [
        label_name for _, label_name in POSITION_FACTOR_COLS
    ]
    header = (
        f"{'Position':<36s} {'Type':<22s}"
        + "".join(f"{column:>12s}" for column in position_cols)
    )
    print(header)
    print("-" * len(header))
    for row in base_attr["positions"]:
        line = f"{row['position_id']:<36s} {row['type']:<22s}"
        line += f"{row['total_pnl']:>12,.0f}"
        for _, label_name in POSITION_FACTOR_COLS:
            line += f"{row[label_name]:>12,.0f}"
        print(line)


Per-position attribution detail skipped.


### Attribution aggregation boundary

Portfolio factor totals above come directly from Rust. The nested position JSON is retained for drill-down only; this notebook intentionally does not recompute financial aggregates by instrument type in Python.

In [ ]:
if base_attr is None:
    print("Nested attribution detail skipped.")
else:
    print(
        f"Nested detail contains {len(base_attr['positions'])} Rust-attributed positions; "
        "portfolio finance totals are read only from PortfolioAttribution."
    )


Nested attribution detail skipped.


### Opt-in attribution at configured scales

In [ ]:
if RUN_ATTRIBUTION_BENCHMARK and RUN_LARGE_BENCHMARK:
    attr_500 = run_portfolio_attribution(
        obj_500, f"Configured scale ({n_500})"
    )
    print_attribution_summary(
        attr_500, f"{n_500}-Position Portfolio Attribution"
    )
else:
    attr_500 = base_attr
    print("Configured-scale attribution skipped; enable attribution and large benchmarks to run it.")


Configured-scale attribution skipped; enable attribution and large benchmarks to run it.


In [ ]:
if RUN_ATTRIBUTION_BENCHMARK and RUN_LARGE_BENCHMARK:
    attr_3000 = run_portfolio_attribution(obj_3000, "Large (3,000)")
    print_attribution_summary(attr_3000, "3,000-Position Portfolio Attribution")
else:
    attr_3000 = attr_500
    print("3,000-position attribution skipped; enable attribution and large benchmarks to run it.")


3,000-position attribution skipped; enable attribution and large benchmarks to run it.


In [ ]:
if RUN_ATTRIBUTION_BENCHMARK and RUN_XL_BENCHMARK:
    attr_25k = run_portfolio_attribution(obj_25k, "XL (25,000)")
    print_attribution_summary(attr_25k, "25,000-Position Portfolio Attribution")
else:
    attr_25k = attr_3000
    print("25,000-position attribution skipped; enable attribution and XL benchmarks to run it.")


25,000-position attribution skipped; enable attribution and XL benchmarks to run it.


### Attribution throughput summary

In [ ]:
if base_attr is None:
    print("P&L attribution throughput skipped.")
else:
    print("P&L Attribution throughput (MetricsBased method)")
    print("=" * 70)
    rows = [(f"Base ({n_priced})", base_attr, base_attr_count)]
    if RUN_LARGE_BENCHMARK:
        rows.append((f"Configured scale ({n_500})", attr_500, n_500))
        rows.append(("Large (3,000)", attr_3000, 3000))
    if RUN_XL_BENCHMARK:
        rows.append(("XL (25,000)", attr_25k, 25_000))
    for label, result, count in rows:
        elapsed = result["elapsed"]
        total = result["totals"]["total_pnl"]
        rate = count / elapsed if elapsed > 0 else float("inf")
        print(
            f"  {label:20s}  {count:5d} positions  {elapsed:7.3f}s  "
            f"{rate:>6,.0f} pos/s  P&L={total:>18,.2f} USD"
        )


P&L attribution throughput skipped.


## 12. Historical & stress scenarios

Apply three scenarios to the portfolio and compare the P&L impact:

| Scenario | Description |
|---|---|
| **GFC 2008** | Rates -200bp (bull steepener), IG credit +150bp, HY credit +400bp, equity -50%, vol +200% |
| **COVID 2020** | Rates -150bp, IG credit +100bp, HY credit +300bp, equity -34%, vol +250% |
| **Static: Rates +200bp / Credit widen** | Parallel rate rise +200bp, IG credit +100bp, HY credit +500bp |

Each scenario is built as a `ScenarioSpec`, applied to the base market via `apply_scenario_to_market`, then the portfolio is revalued under the stressed market.

In [ ]:
gfc_ops = json.dumps([
    {"kind": "curve_parallel_bp", "curve_kind": "discount", "curve_id": "USD-OIS", "bp": -200.0},
    {"kind": "curve_node_bp", "curve_kind": "discount", "curve_id": "USD-OIS",
     "nodes": [["2Y", -50.0], ["5Y", 0.0], ["10Y", 50.0]], "match_mode": "interpolate"},
    {"kind": "curve_parallel_bp", "curve_kind": "discount", "curve_id": "EUR-OIS", "bp": -150.0},
    {"kind": "curve_parallel_bp", "curve_kind": "discount", "curve_id": "USD-IG", "bp": 100.0},
    {"kind": "curve_parallel_bp", "curve_kind": "discount", "curve_id": "USD-CREDIT-BBB", "bp": 600.0},
    {"kind": "curve_parallel_bp", "curve_kind": "par_cds", "curve_id": "CORP-HAZARD",
     "discount_curve_id": "USD-OIS", "bp": 400.0},
    {"kind": "curve_parallel_bp", "curve_kind": "par_cds", "curve_id": "CDX-HAZ",
     "discount_curve_id": "USD-OIS", "bp": 150.0},
    {"kind": "equity_price_pct", "ids": ["TECH", "AAPL-SPOT", "SPX-SPOT"], "pct": -50.0},
    {"kind": "vol_surface_parallel_pct", "surface_kind": "equity", "surface_id": "TECH-VOL", "pct": 200.0},
    {"kind": "vol_surface_parallel_pct", "surface_kind": "credit", "surface_id": "CDS-SPREAD-VOL", "pct": 150.0},
    {"kind": "vol_surface_parallel_pct", "surface_kind": "swaption", "surface_id": "USD-SWPNVOL", "pct": 100.0},
    {"kind": "market_fx_pct", "base": "EUR", "quote": "USD", "pct": -10.0},
])
gfc_spec = build_scenario_spec(
    "gfc_2008", gfc_ops,
    name="Global Financial Crisis 2008",
    description="Lehman collapse: rates rally, credit blowout, equity crash, vol spike",
)

covid_ops = json.dumps([
    {"kind": "curve_parallel_bp", "curve_kind": "discount", "curve_id": "USD-OIS", "bp": -150.0},
    {"kind": "curve_node_bp", "curve_kind": "discount", "curve_id": "USD-OIS",
     "nodes": [["2Y", -30.0], ["10Y", 10.0]], "match_mode": "interpolate"},
    {"kind": "curve_parallel_bp", "curve_kind": "discount", "curve_id": "EUR-OIS", "bp": -100.0},
    {"kind": "curve_parallel_bp", "curve_kind": "discount", "curve_id": "USD-IG", "bp": 50.0},
    {"kind": "curve_parallel_bp", "curve_kind": "discount", "curve_id": "USD-CREDIT-BBB", "bp": 450.0},
    {"kind": "curve_parallel_bp", "curve_kind": "par_cds", "curve_id": "CORP-HAZARD",
     "discount_curve_id": "USD-OIS", "bp": 300.0},
    {"kind": "curve_parallel_bp", "curve_kind": "par_cds", "curve_id": "CDX-HAZ",
     "discount_curve_id": "USD-OIS", "bp": 100.0},
    {"kind": "equity_price_pct", "ids": ["TECH", "AAPL-SPOT", "SPX-SPOT"], "pct": -34.0},
    {"kind": "vol_surface_parallel_pct", "surface_kind": "equity", "surface_id": "TECH-VOL", "pct": 250.0},
    {"kind": "vol_surface_parallel_pct", "surface_kind": "credit", "surface_id": "CDS-SPREAD-VOL", "pct": 200.0},
    {"kind": "vol_surface_parallel_pct", "surface_kind": "swaption", "surface_id": "USD-SWPNVOL", "pct": 120.0},
    {"kind": "market_fx_pct", "base": "EUR", "quote": "USD", "pct": -5.0},
])
covid_spec = build_scenario_spec(
    "covid_2020", covid_ops,
    name="COVID 2020 Market Shock",
    description="Pandemic panic: emergency easing, credit widening, equity crash, vol spike",
)

static_ops = json.dumps([
    {"kind": "curve_parallel_bp", "curve_kind": "discount", "curve_id": "USD-OIS", "bp": 200.0},
    {"kind": "curve_parallel_bp", "curve_kind": "discount", "curve_id": "EUR-OIS", "bp": 200.0},
    {"kind": "curve_parallel_bp", "curve_kind": "forward", "curve_id": "USD-SOFR-3M", "bp": 200.0},
    {"kind": "curve_parallel_bp", "curve_kind": "discount", "curve_id": "USD-IG", "bp": 300.0},
    {"kind": "curve_parallel_bp", "curve_kind": "discount", "curve_id": "USD-CREDIT-BBB", "bp": 700.0},
    {"kind": "curve_parallel_bp", "curve_kind": "par_cds", "curve_id": "CORP-HAZARD",
     "discount_curve_id": "USD-OIS", "bp": 500.0},
    {"kind": "curve_parallel_bp", "curve_kind": "par_cds", "curve_id": "CDX-HAZ",
     "discount_curve_id": "USD-OIS", "bp": 100.0},
])
static_spec = build_scenario_spec(
    "rates_up_credit_wide", static_ops,
    name="Rates +200bp / Credit Widen (IG +100, HY +500)",
    description="Static stress: parallel rate rise with IG and HY credit spread widening",
)

SCENARIOS = [
    ("GFC 2008", gfc_spec),
    ("COVID 2020", covid_spec),
    ("Rates +200 / Credit Wide", static_spec),
]

print(f"Built {len(SCENARIOS)} scenario specs")

Built 3 scenario specs


### Scenario helper: one Rust call per scenario

`portfolio.scenario_pnl(portfolio, scenario_json, market)` replaces the
apply-then-revalue-then-diff dance. Rust applies the scenario to the market, values the
book against both the unshocked and shocked markets, and returns the base-currency
difference as `total` plus a `by_position` ladder (all `Money`), together with the
scenario `ApplicationReport` (operations applied, warnings). Positions the scenario adds
or removes are zero-filled on the missing side, so `by_position` always sums to `total`.

In [ ]:
def run_scenario(scenario_name: str, scenario_spec: str, portfolio_json: str,
                 base_valuation_json: str) -> dict:
    """Read the Rust-computed scenario P&L ladder for one scenario."""
    t_start = time.perf_counter()
    # One call: Rust applies the shocks, revalues the book, and differences the
    # stressed valuation against the base — total and per position, in base ccy.
    pnl_json, report_json = scenario_pnl(portfolio_json, scenario_spec, MARKET_JSON)
    elapsed = time.perf_counter() - t_start

    pnl = json.loads(pnl_json)
    report = json.loads(report_json)

    total_pnl = float(pnl["total"]["amount"])
    position_pnls = {
        position_id: float(money["amount"]) for position_id, money in pnl["by_position"].items()
    }

    # Base PV is the valuation Rust already produced upstream; the stressed level is
    # that PV plus the Rust P&L — a display identity, not a re-derived valuation.
    base_pv = PortfolioValuation.from_json(base_valuation_json).total_value

    return {
        "name": scenario_name,
        "ops_applied": report["operations_applied"],
        "warnings": report["warnings"],
        "base_pv": base_pv,
        "stressed_pv": base_pv + total_pnl,
        "total_pnl": total_pnl,
        "pnl_pct": total_pnl / abs(base_pv) * 100 if abs(base_pv) > 1e-6 else 0.0,
        "position_pnls": position_pnls,
        "elapsed": elapsed,
        "num_positions": len(position_pnls),
    }

### Configured base portfolio under each scenario

In [ ]:
base_scenario_results = []
for name, spec in SCENARIOS:
    r = run_scenario(name, spec, base_portfolio_json, val_result)
    base_scenario_results.append(r)
    print(f"{name:30s}  ops={r['ops_applied']:2d}  "
          f"P&L={r['total_pnl']:>16,.0f} USD  ({r['pnl_pct']:+.1f}%)  "
          f"{r['elapsed']:.3f}s  warnings={len(r['warnings'])}")

GFC 2008                        ops=14  P&L=          56,163 USD  (+4.6%)  0.302s  warnings=0
COVID 2020                      ops=14  P&L=          42,230 USD  (+3.4%)  0.306s  warnings=0
Rates +200 / Credit Wide        ops= 7  P&L=         477,715 USD  (+38.8%)  0.290s  warnings=0


### Per-position scenario P&L (base portfolio, top movers)

In [ ]:
scenario_names = [r["name"] for r in base_scenario_results]
header = f"{'Position':<14s}" + "".join(f"{n:>22s}" for n in scenario_names)
print("Top 10 P&L movers (sorted by worst GFC loss)")
print(header)
print("-" * len(header))

all_pos_ids = list(base_scenario_results[0]["position_pnls"].keys())
sorted_pos = sorted(all_pos_ids, key=lambda p: base_scenario_results[0]["position_pnls"].get(p, 0))

for pos_id in sorted_pos[:10]:
    line = f"{pos_id:<14s}"
    for r in base_scenario_results:
        pnl = r["position_pnls"].get(pos_id, 0)
        line += f"{pnl:>22,.0f}"
    print(line)

Top 10 P&L movers (sorted by worst GFC loss)
Position                    GFC 2008            COVID 2020Rates +200 / Credit Wide
----------------------------------------------------------------------------------
POS-2 (AAPL 100sh)                -9,250                -6,290                     0
POS-1 (IRS Pay 3Y 4.0%)                 7,278                 5,298               532,173
POS-0 (Fixed 3Y 4.0%)                58,134                43,222               -54,458


### Scale scenarios to the configured scale, 3,000, and 25,000 positions

In [ ]:
base_position_count = len(PortfolioValuation.from_json(val_result))
all_scenario_results = {
    str(base_position_count): base_scenario_results
}
scenario_inputs = []
if RUN_LARGE_BENCHMARK:
    scenario_inputs.extend(
        [
            (portfolio_500_json, val_500),
            (portfolio_3000_json, val_3000),
        ]
    )
if RUN_XL_BENCHMARK:
    scenario_inputs.append((portfolio_25k_json, val_25k))
if not scenario_inputs:
    print("Scale scenario runs skipped; enable large or XL benchmarks to run them.")

for portfolio_json, base_value_json in scenario_inputs:
    results = []
    for name, spec in SCENARIOS:
        result = run_scenario(
            name, spec, portfolio_json, base_value_json
        )
        results.append(result)
    count = len(PortfolioValuation.from_json(base_value_json))
    all_scenario_results[str(count)] = results
    print(f"\n{'=' * 80}")
    print(f"  {count:,d}-position portfolio")
    print("=" * 80)
    for result in results:
        print(
            f"  {result['name']:30s}  "
            f"P&L={result['total_pnl']:>18,.0f} USD  "
            f"({result['pnl_pct']:+.1f}%)  {result['elapsed']:.3f}s"
        )


Scale scenario runs skipped; enable large or XL benchmarks to run them.


### Scenario summary across all portfolio sizes

In [ ]:
print("Scenario P&L impact across portfolio sizes")
print("=" * 100)
header = f"{'Scenario':30s}  {'Positions':>9s}  {'Base PV':>18s}  {'Stressed PV':>18s}  {'P&L':>18s}  {'%':>7s}  {'Time':>7s}"
print(header)
print("-" * 100)

size_labels = list(all_scenario_results)

for size_label in size_labels:
    for r in all_scenario_results[size_label]:
        print(f"{r['name']:30s}  {r['num_positions']:>9,d}  "
              f"{r['base_pv']:>18,.0f}  {r['stressed_pv']:>18,.0f}  "
              f"{r['total_pnl']:>18,.0f}  {r['pnl_pct']:>+6.1f}%  {r['elapsed']:>6.2f}s")
    print()

Scenario P&L impact across portfolio sizes
Scenario                        Positions             Base PV         Stressed PV                 P&L        %     Time
----------------------------------------------------------------------------------------------------
GFC 2008                                3           1,230,138           1,286,300              56,163    +4.6%    0.30s
COVID 2020                              3           1,230,138           1,272,368              42,230    +3.4%    0.31s
Rates +200 / Credit Wide                3           1,230,138           1,707,853             477,715   +38.8%    0.29s



## Takeaways

- **Seventeen instrument types** share a single `MarketContext` snapshot and a single `value_portfolio` call.
- Every position automatically receives **bucketed DV01**, **bucketed CS01**, **delta**, **gamma**, **vega**, **rho**, **pv01**, and **theta** where applicable -- the standard risk set an investment group monitors.
- `aggregate_metrics` consolidates position-level sensitivities into portfolio totals and entity breakdowns, handling FX conversion for multi-currency books.
- The JSON-based portfolio spec scales linearly: position factory functions parameterize maturity, coupon, strike, and side to generate diverse but valid instruments.
- **1-day P&L attribution** decomposes overnight MTM changes into carry, rates, credit, vol, FX, and cross-factor contributions with a small residual, running at scale across the full book.
- **Historical and stress scenarios** (GFC, COVID, rates+credit) apply parameterised shocks via `apply_scenario_to_market` and revalue the entire portfolio in a single call, producing scenario P&L at all three scales.
- At 25,000 positions the Rust engine handles valuation, attribution, and scenario analysis in seconds, making full-book risk reporting practical even at institutional scale.
- Structured credit (CLO/ABS) instruments are the heaviest per-position; the throughput numbers are blended averages across all types.